# 피처 엔지니어링
차트 기반 파생변수 생성 + 피처 중요도 분석

In [ ]:
# 셀 1 - 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from dotenv import load_dotenv
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

load_dotenv('../../collect_data/hyerim/.env')
client = MongoClient(os.getenv('MONGO_URI'))
db = client['data_hyerim']
print('MongoDB 연결 완료')

In [ ]:
# 셀 2 - 데이터 불러오기
cursor = db['stock_ohlcv_kospi200'].find({}, {'_id': 0})
df = pd.DataFrame(list(cursor))
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['stock_code', 'date']).reset_index(drop=True)

print(f'shape: {df.shape}')
print(f'종목 수: {df["stock_code"].nunique()}')
print(f'기간: {df["date"].min()} ~ {df["date"].max()}')
df.head()

In [ ]:
# 셀 3 - 피처 엔지니어링 함수 정의

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values('date').reset_index(drop=True)

    # -------------------------------------------------------
    # 이동평균선
    # -------------------------------------------------------
    for w in [5, 10, 20, 60, 120, 240]:
        df[f'ma{w}'] = df['close'].rolling(w).mean()

    # 정배열 여부
    df['is_uptrend'] = (
        (df['ma5'] > df['ma10']) &
        (df['ma10'] > df['ma20']) &
        (df['ma20'] > df['ma60'])
    ).astype(int)

    # 역배열 여부
    df['is_downtrend'] = (
        (df['ma5'] < df['ma10']) &
        (df['ma10'] < df['ma20']) &
        (df['ma20'] < df['ma60'])
    ).astype(int)

    # 이평선 간격
    df['ma_spread_5_60']  = (df['ma5']  - df['ma60']) / (df['ma60'] + 1)
    df['ma_spread_10_60'] = (df['ma10'] - df['ma60']) / (df['ma60'] + 1)

    # 60일선 괴리율
    df['ma60_gap'] = (df['close'] - df['ma60']) / (df['ma60'] + 1)

    # 10일선 이탈 여부 (종가 기준)
    df['ma10_break'] = (df['close'] < df['ma10']).astype(int)

    # 골든크로스 / 데드크로스
    df['ma_cross_5_10']      = ((df['ma5'] > df['ma10']) & (df['ma5'].shift(1) <= df['ma10'].shift(1))).astype(int)
    df['ma_cross_5_10_dead'] = ((df['ma5'] < df['ma10']) & (df['ma5'].shift(1) >= df['ma10'].shift(1))).astype(int)

    # 캔들이 60일선 위 여부
    df['candle_above_ma60'] = (df['close'] > df['ma60']).astype(int)

    # -------------------------------------------------------
    # RSI (9일)
    # -------------------------------------------------------
    period = 9
    delta  = df['close'].diff()
    gain   = delta.clip(lower=0).ewm(com=period - 1, adjust=False).mean()
    loss   = (-delta.clip(upper=0)).ewm(com=period - 1, adjust=False).mean()
    rs     = gain / (loss + 1e-9)
    df['rsi_9'] = 100 - (100 / (1 + rs))

    df['rsi_above_70'] = (df['rsi_9'] > 70).astype(int)
    df['rsi_above_50'] = (df['rsi_9'] > 50).astype(int)
    df['rsi_exit_70']  = ((df['rsi_9'] < 70) & (df['rsi_9'].shift(1) >= 70)).astype(int)

    # RSI 이동평균선
    df['rsi_ma'] = df['rsi_9'].rolling(9).mean()

    # RSI가 이동평균선 터치한 횟수 (최근 5일 기준)
    rsi_touch = ((df['rsi_9'] - df['rsi_ma']).abs() < 1.5).astype(int)
    df['rsi_touch_ma_count'] = rsi_touch.rolling(5).sum()

    # -------------------------------------------------------
    # MACD (12 / 25 / 60)
    # -------------------------------------------------------
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema25 = df['close'].ewm(span=25, adjust=False).mean()
    df['macd']        = ema12 - ema25
    df['macd_signal'] = df['macd'].ewm(span=60, adjust=False).mean()
    df['macd_hist']   = df['macd'] - df['macd_signal']

    df['macd_uptrend']   = (df['macd'] > df['macd_signal']).astype(int)
    df['macd_above_zero'] = (df['macd'] > 0).astype(int)
    df['macd_near_zero']  = (df['macd'].abs() < df['macd'].rolling(20).std() * 0.3).astype(int)

    # -------------------------------------------------------
    # 볼린저밴드 (20일, 2sigma)
    # -------------------------------------------------------
    df['bb_mid']   = df['close'].rolling(20).mean()
    df['bb_std']   = df['close'].rolling(20).std()
    df['bb_upper'] = df['bb_mid'] + 2 * df['bb_std']
    df['bb_lower'] = df['bb_mid'] - 2 * df['bb_std']
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (df['bb_mid'] + 1)
    df['bb_pct']   = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1)

    df['bb_upper_break']  = (df['close'] > df['bb_upper']).astype(int)
    df['bb_lower_support'] = (df['close'] <= df['bb_lower']).astype(int)

    # -------------------------------------------------------
    # 일목균형표
    # -------------------------------------------------------
    high9  = df['high'].rolling(9).max()
    low9   = df['low'].rolling(9).min()
    high26 = df['high'].rolling(26).max()
    low26  = df['low'].rolling(26).min()
    high52 = df['high'].rolling(52).max()
    low52  = df['low'].rolling(52).min()

    df['ichi_conversion'] = (high9  + low9)  / 2
    df['ichi_base']       = (high26 + low26) / 2
    df['ichi_span_a']     = (df['ichi_conversion'] + df['ichi_base']) / 2
    df['ichi_span_b']     = (high52 + low52) / 2

    cloud_top    = df[['ichi_span_a', 'ichi_span_b']].max(axis=1)
    cloud_bottom = df[['ichi_span_a', 'ichi_span_b']].min(axis=1)

    df['ichi_above_cloud']    = (df['close'] > cloud_top).astype(int)
    df['ichi_below_cloud']    = (df['close'] < cloud_bottom).astype(int)
    df['ichi_in_cloud']       = ((df['close'] >= cloud_bottom) & (df['close'] <= cloud_top)).astype(int)
    df['cloud_green']         = (df['ichi_span_a'] > df['ichi_span_b']).astype(int)
    df['cloud_thickness']     = (cloud_top - cloud_bottom) / (df['close'] + 1)
    df['ichi_conversion_cross'] = (
        (df['ichi_conversion'] > df['ichi_base']) &
        (df['ichi_conversion'].shift(1) <= df['ichi_base'].shift(1))
    ).astype(int)

    # -------------------------------------------------------
    # 피보나치 (직전 상승파 기준)
    # -------------------------------------------------------
    roll_high = df['high'].rolling(60).max()
    roll_low  = df['low'].rolling(60).min()
    fib_range = roll_high - roll_low

    fib_786 = roll_high - fib_range * 0.786
    fib_618 = roll_high - fib_range * 0.618
    fib_500 = roll_high - fib_range * 0.500

    df['fib_786_support'] = (df['close'].between(fib_786 * 0.99, fib_786 * 1.01)).astype(int)
    df['fib_618_level']   = (df['close'].between(fib_618 * 0.99, fib_618 * 1.01)).astype(int)
    df['fib_500_level']   = (df['close'].between(fib_500 * 0.99, fib_500 * 1.01)).astype(int)

    # 전고점 돌파 여부
    prev_high = df['high'].shift(1).rolling(20).max()
    df['prev_high_break'] = (df['close'] > prev_high).astype(int)
    df['prev_high_fail']  = (
        (df['high'] >= prev_high * 0.99) & (df['close'] < prev_high)
    ).astype(int)

    # -------------------------------------------------------
    # 추세 / 지지저항
    # -------------------------------------------------------
    df['higher_high'] = (df['high'] > df['high'].shift(1)).astype(int)
    df['higher_low']  = (df['low']  > df['low'].shift(1)).astype(int)
    df['lower_high']  = (df['high'] < df['high'].shift(1)).astype(int)

    # 쌍바닥: 직전 저점보다 현재 저점이 높은 경우
    prev_low = df['low'].rolling(10).min().shift(1)
    df['double_bottom']      = (df['low'] > prev_low).astype(int)
    df['right_bottom_higher'] = df['double_bottom']  # 동일 조건

    # -------------------------------------------------------
    # 캔들 패턴
    # -------------------------------------------------------
    candle_range = df['high'] - df['low'] + 1
    df['body_ratio']       = abs(df['close'] - df['open']) / candle_range
    df['is_bullish']       = (df['close'] > df['open']).astype(int)
    df['is_bearish']       = (df['close'] < df['open']).astype(int)
    df['is_doji']          = (df['body_ratio'] < 0.1).astype(int)
    df['upper_wick_ratio'] = (df['high'] - df[['open','close']].max(axis=1)) / candle_range
    df['lower_wick_ratio'] = (df[['open','close']].min(axis=1) - df['low']) / candle_range

    prev_close = df['close'].shift(1)
    df['gap_up']        = (df['open'] > prev_close * 1.005).astype(int)
    df['gap_down']      = (df['open'] < prev_close * 0.995).astype(int)
    df['gap_fill_rate'] = ((df['close'] - df['open']) / (prev_close - df['open'] + 1e-9)).clip(-1, 1)

    # -------------------------------------------------------
    # 거래량
    # -------------------------------------------------------
    vol_ma20 = df['volume'].rolling(20).mean()
    df['vol_ratio_20']          = df['volume'] / (vol_ma20 + 1)
    df['vol_spike']             = (df['vol_ratio_20'] >= 2).astype(int)
    df['vol_up_with_price_up']  = ((df['volume'] > vol_ma20) & (df['close'] > df['close'].shift(1))).astype(int)
    df['vol_down_with_price_up'] = ((df['volume'] < vol_ma20) & (df['close'] > df['close'].shift(1))).astype(int)

    return df

print('피처 함수 정의 완료')

In [ ]:
# 셀 4 - 삼성전자 기준 피처 계산 + 라벨 생성
df_ss = df[df['stock_code'] == '005930'].copy()
df_ss = add_features(df_ss)

# 5일 후 수익률 기반 라벨 (매수=1, 매도=2, 관망=0)
df_ss['future_return'] = df_ss['close'].shift(-5) / df_ss['close'] - 1
df_ss['target'] = 0
df_ss.loc[df_ss['future_return'] >  0.02, 'target'] = 1  # 매수
df_ss.loc[df_ss['future_return'] < -0.02, 'target'] = 2  # 매도

df_ss = df_ss.dropna().reset_index(drop=True)

print(f'피처 계산 완료: {df_ss.shape}')
print(f'target 분포:\n{df_ss["target"].value_counts()}')
df_ss.head()

In [ ]:
# 셀 5 - 피처 컬럼 목록
feature_cols = [
    # 이동평균선
    'is_uptrend', 'is_downtrend', 'ma_spread_5_60', 'ma_spread_10_60',
    'ma60_gap', 'ma10_break', 'ma_cross_5_10', 'ma_cross_5_10_dead', 'candle_above_ma60',
    # RSI
    'rsi_9', 'rsi_above_70', 'rsi_above_50', 'rsi_exit_70', 'rsi_touch_ma_count',
    # MACD
    'macd', 'macd_signal', 'macd_hist', 'macd_uptrend', 'macd_above_zero', 'macd_near_zero',
    # 볼린저밴드
    'bb_pct', 'bb_width', 'bb_upper_break', 'bb_lower_support',
    # 일목균형표
    'ichi_above_cloud', 'ichi_below_cloud', 'ichi_in_cloud',
    'cloud_green', 'cloud_thickness', 'ichi_conversion_cross',
    # 피보나치
    'fib_786_support', 'fib_618_level', 'fib_500_level',
    'prev_high_break', 'prev_high_fail',
    # 추세/지지저항
    'higher_high', 'higher_low', 'lower_high', 'double_bottom',
    # 캔들
    'body_ratio', 'is_bullish', 'is_bearish', 'is_doji',
    'upper_wick_ratio', 'lower_wick_ratio',
    'gap_up', 'gap_down', 'gap_fill_rate',
    # 거래량
    'vol_ratio_20', 'vol_spike', 'vol_up_with_price_up', 'vol_down_with_price_up',
]

X = df_ss[feature_cols]
y = df_ss['target']
print(f'피처 수: {len(feature_cols)}')
print(f'X shape: {X.shape}')

In [ ]:
# 셀 6 - 1) 상관계수
corr_result = df_ss[feature_cols + ['target']].corr()['target'].abs().drop('target').sort_values(ascending=False)

plt.figure(figsize=(10, 12))
sns.barplot(x=corr_result.values, y=corr_result.index, palette='Blues_r')
plt.title('피처별 상관계수 (절대값) vs target')
plt.xlabel('상관계수 절대값')
plt.tight_layout()
plt.show()
print(corr_result)

In [ ]:
# 셀 7 - 2) Mutual Information
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_result = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 12))
sns.barplot(x=mi_result.values, y=mi_result.index, palette='Greens_r')
plt.title('피처별 Mutual Information (비선형 관계)')
plt.xlabel('MI Score')
plt.tight_layout()
plt.show()
print(mi_result)

In [ ]:
# 셀 8 - 3) RandomForest Feature Importance
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X, y)

rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 12))
sns.barplot(x=rf_importance.values, y=rf_importance.index, palette='Oranges_r')
plt.title('RandomForest Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()
print(rf_importance)

In [ ]:
# 셀 9 - 종합 점수 히트맵
importance_df = pd.DataFrame({
    '상관계수':   corr_result,
    'MutualInfo': mi_result,
    'RF중요도':   rf_importance,
})

importance_df['총점'] = (
    importance_df['상관계수'].rank() +
    importance_df['MutualInfo'].rank() +
    importance_df['RF중요도'].rank()
)
importance_df = importance_df.sort_values('총점', ascending=False)

plt.figure(figsize=(10, 14))
sns.heatmap(
    importance_df[['상관계수', 'MutualInfo', 'RF중요도']],
    annot=True, fmt='.4f', cmap='YlOrRd', linewidths=0.5
)
plt.title('피처 중요도 종합 히트맵 (3가지 기준)')
plt.tight_layout()
plt.show()

print('\n최종 피처 우선순위 (상위 20개):')
print(importance_df[['총점']].head(20))

In [ ]:
# 셀 10 - 상위 피처 간 상관관계 히트맵
top_features = importance_df.index[:8].tolist()

plt.figure(figsize=(10, 8))
sns.heatmap(
    df_ss[top_features + ['target']].corr(),
    annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5
)
plt.title('상위 피처 간 상관관계 히트맵')
plt.tight_layout()
plt.show()

In [ ]:
# 셀 11 - target별 상위 피처 분포 boxplot
fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for ax, col in zip(axes.flatten(), top_features):
    sns.boxplot(
        data=df_ss, x='target', y=col, ax=ax,
        palette=['#4C72B0', '#55A868', '#DD8452']
    )
    ax.set_title(col)
    ax.set_xlabel('target (0=관망, 1=매수, 2=매도)')

plt.suptitle('target별 상위 피처 분포', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# 셀 12 - 전종목 피처 계산 후 저장
all_features = []

for code, group in df.groupby('stock_code'):
    try:
        feat = add_features(group)
        feat['future_return'] = feat['close'].shift(-5) / feat['close'] - 1
        feat['target'] = 0
        feat.loc[feat['future_return'] >  0.02, 'target'] = 1
        feat.loc[feat['future_return'] < -0.02, 'target'] = 2
        all_features.append(feat)
    except Exception as e:
        print(f'[skip] {code}: {e}')

df_all = pd.concat(all_features, ignore_index=True).dropna(subset=feature_cols + ['target'])
print(f'전종목 피처 계산 완료: {df_all.shape}')
df_all.head()